# MP24 — Spatial Attention from Scratch

Spatial attention pools across feature channels, then predicts a multiplier at each location. A bright mask does not establish a meaningful object or correct explanation.

## Mathematics and worked example

Mean and maximum across C produce two N×1×H×W maps. Concatenate them, apply a padded 7×7 convolution, and sigmoid to get N×1×H×W. This has 2×7×7+1=99 parameters. For two channels with values 2 and 4 at one pixel, pooled mean=3 and maximum=4. If the spatial gate is 0.25, that pixel becomes [0.5,1]. Zero padding affects border masks.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from rssr.attention_labs import InspectableAttention
torch.manual_seed(23)
x=torch.rand(1,8,5,7)
g=InspectableAttention(8)
y,c,s=g(x,'spatial')
assert s.shape==(1,1,5,7)
assert sum(p.numel() for p in g.spatial.parameters())==99
torch.testing.assert_close(y,x*s)
print('Gate range',s.min().item(),s.max().item())

## Executed experiment and reproducibility

Run `python scripts/attention_lessons_experiment.py` from the repository root. This is same-grid RGB reconstruction, not super-resolution. A tiny shape fixture precedes training on four NAIP scenes, with four separate validation scenes, 16 fixed 32×32 patches each. USDA/OpenSR NAIP imagery is public domain; values are stored DN/10000, not claimed calibrated reflectance. No new imagery download or final-test access is required.

CPU float32, two threads, seed 23, Adam 0.001, batch four, 300 updates, MSE. Models share identical head/tail initialization and minibatch sequence. Evaluate the fixed last step, with no checkpoint selection. Validation PSNR uses range one, border two, equal-size patches and equal scene weighting. Inactive gate modules remain allocated to preserve initialization; the report separately counts active and allocated parameters. Inference timing uses five warmups and twenty observations on a 32×32 input; it is not whole-app latency.

The first figure compares separately trained reconstructions under one reference contrast scale. The second applies alternative gate modes to identical features from the trained combined model, plus synthetic step-edge/checkerboard fixtures. All mask heatmaps share [0,1]; these are not semantic labels. Training curves show sampled minibatch loss, not a full-dataset estimate. Read the actual measured table below; preserve unfavorable outcomes.


In [ ]:
r=json.loads((ROOT/'reports/mp23-mp25-attention.json').read_text())
for row in r['results']:
 print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import display,Image
for name in ['mp23-mp25-reconstruction.png','mp23-mp25-attention.png']:
 display(Image(filename=str(ROOT/'reports'/name)))

## Practice

1. Why use two pooled maps?
2. Is an edge heatmap segmentation accuracy?
3. What is broadcast over channels?

## Worked solutions

1. Mean captures average response; maximum captures a strong response. Their usefulness still needs measurement.
2. No. There are no semantic labels or segmentation evaluation here.
3. The same scalar at a spatial location multiplies every feature channel.

## Limitations

One seed and a short reconstruction experiment cannot establish SR superiority, causality, explainability or geographic generalization. Equal update count is not equal computation. Learner mastery is unassessed. Frozen research weights and test results are untouched.

[Authoritative brief](../stage_06_attention/MP24_spatial_attention_from_scratch/README.md) · [Local/Colab setup](../README.md)